In [8]:
!pip install experta -q

In [9]:
import collections, collections.abc
for _n in ("Mapping", "MutableMapping", "Sequence", "Iterable", "Callable"):
    if not hasattr(collections, _n):
        setattr(collections, _n, getattr(collections.abc, _n))

from dataclasses import dataclass, field
from typing import List
from experta import KnowledgeEngine, Fact, Rule, MATCH, TEST, DefFacts


@dataclass
class Frame:
    key: str; label: str; unit: str; limit: float; direction: str
    band_low: float = None; band_high: float = None
    contaminant: str = ""; health_effect: str = ""
    sources: List[str] = field(default_factory=list)
    treatments: List[str] = field(default_factory=list)
    guideline: str = ""


# Knowledge base built from the WHO and US EPA drinking-water guidelines
KB = {
 "nitrate": Frame("nitrate","Nitrate (as NO3)","mg/L",50.0,"max",
    contaminant="Nitrate_Exceedance",
    health_effect="Methemoglobinemia (blue-baby syndrome) in infants",
    sources=["Agricultural_runoff","Septic_leakage","Fertiliser_use"],
    treatments=["Ion_exchange","Reverse_osmosis","Alternative_supply"],
    guideline="WHO 50 mg/L"),
 "arsenic": Frame("arsenic","Arsenic","mg/L",0.01,"max",
    contaminant="Arsenic_Exceedance",
    health_effect="Skin lesions and long-term cancer risk (skin, bladder, lung)",
    sources=["Natural_geological","Industrial_discharge","Mining_activity"],
    treatments=["Reverse_osmosis","Activated_alumina","Ion_exchange","Coagulation_filtration"],
    guideline="WHO / EPA 0.01 mg/L"),
 "lead": Frame("lead","Lead","mg/L",0.01,"max",
    contaminant="Lead_Exceedance",
    health_effect="Neurodevelopmental harm in children, kidney damage",
    sources=["Corroded_plumbing","Lead_pipes_or_solder","Brass_fixtures"],
    treatments=["Corrosion_control","Reverse_osmosis","Replace_plumbing"],
    guideline="WHO 0.01 mg/L (EPA action level 0.015 mg/L)"),
 "fluoride": Frame("fluoride","Fluoride","mg/L",1.5,"max",
    contaminant="Fluoride_Exceedance",
    health_effect="Dental and, at higher exposure, skeletal fluorosis",
    sources=["Natural_geological","Industrial_discharge"],
    treatments=["Activated_alumina","Reverse_osmosis","Bone_char_filtration"],
    guideline="WHO 1.5 mg/L"),
 "turbidity": Frame("turbidity","Turbidity","NTU",5.0,"max",
    contaminant="High_Turbidity",
    health_effect="Particles can shield microbes and reduce disinfection",
    sources=["Surface_runoff","Sediment","Inadequate_filtration"],
    treatments=["Coagulation_filtration","Sedimentation"],
    guideline="WHO acceptability 5 NTU (target < 1 NTU)"),
 "ph": Frame("ph","pH","",None,"band",band_low=6.5,band_high=8.5,
    contaminant="pH_Out_Of_Range",
    health_effect="No direct health effect, but affects corrosion and taste",
    sources=["Source_water_chemistry","Treatment_imbalance"],
    treatments=["pH_adjustment"],
    guideline="WHO / EPA aesthetic range 6.5 - 8.5"),
 "chlorine": Frame("chlorine","Free chlorine residual","mg/L",0.2,"min",
    contaminant="Low_Chlorine_Residual",
    health_effect="Weak protection against microbial regrowth in the network",
    sources=["Long_distribution_time","Chlorine_demand","Dosing_fault"],
    treatments=["Re_chlorination","Booster_disinfection"],
    guideline="WHO recommends >= 0.2 mg/L at the point of delivery"),
 "coliform": Frame("coliform","Total coliform / E. coli","present/absent",0,"max",
    contaminant="Microbial_Contamination",
    health_effect="Risk of gastrointestinal disease from faecal pathogens",
    sources=["Sewage_ingress","Animal_waste","Biofilm"],
    treatments=["Disinfection_chlorination","UV_treatment","Boil_water_advisory"],
    guideline="WHO / EPA must be absent in 100 mL"),
}


# Fact types used by the rule engine
class Reading(Fact): pass
class Presence(Fact): pass
class Limit(Fact): pass
class Exceedance(Fact): pass


# The forward-chaining production rules
class AquaRules(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.exceedances = []; self.inferences = []; self.trace = []

    @DefFacts()
    def load_limits(self):
        for f in KB.values():
            if f.direction == "band":
                yield Limit(param=f.key, kind="band_low", value=f.band_low)
                yield Limit(param=f.key, kind="band_high", value=f.band_high)
            elif f.direction == "min":
                yield Limit(param=f.key, kind="min", value=f.limit)
            else:
                yield Limit(param=f.key, kind="max", value=f.limit)

    def _flag(self, param, reason):
        if param not in self.exceedances:
            self.exceedances.append(param)
        self.trace.append(reason)
        self.declare(Exceedance(param=param))

    @Rule(Reading(param=MATCH.p, value=MATCH.v),
          Limit(param=MATCH.p, kind="max", value=MATCH.lim), TEST(lambda v, lim: v > lim))
    def over_max(self, p, v, lim):
        self._flag(p, "R1  %s reading %s is above the limit %s -> %s" % (p, v, lim, KB[p].contaminant))

    @Rule(Reading(param=MATCH.p, value=MATCH.v),
          Limit(param=MATCH.p, kind="min", value=MATCH.lim), TEST(lambda v, lim: v < lim))
    def under_min(self, p, v, lim):
        self._flag(p, "R2  %s reading %s is below the minimum %s -> %s" % (p, v, lim, KB[p].contaminant))

    @Rule(Reading(param=MATCH.p, value=MATCH.v),
          Limit(param=MATCH.p, kind="band_low", value=MATCH.lo), TEST(lambda v, lo: v < lo))
    def below_band(self, p, v, lo):
        self._flag(p, "R3  %s reading %s is below the healthy band (< %s) -> %s" % (p, v, lo, KB[p].contaminant))

    @Rule(Reading(param=MATCH.p, value=MATCH.v),
          Limit(param=MATCH.p, kind="band_high", value=MATCH.hi), TEST(lambda v, hi: v > hi))
    def above_band(self, p, v, hi):
        self._flag(p, "R4  %s reading %s is above the healthy band (> %s) -> %s" % (p, v, hi, KB[p].contaminant))

    @Rule(Presence(param=MATCH.p, present=True))
    def presence_positive(self, p):
        self._flag(p, "R5  %s detected as present -> %s" % (p, KB[p].contaminant))

    # ---- combination rules: infer a new fact from several findings ----
    @Rule(Exceedance(param="chlorine"), Exceedance(param="coliform"))
    def low_chlorine_and_coliform(self):
        self.inferences.append(("Recent_microbial_or_biofilm_event",
            "Low chlorine residual together with coliform presence points to recent microbial contamination or a biofilm problem."))
        self.trace.append("R6  chlorine low AND coliform present -> recent microbial / biofilm event")

    @Rule(Exceedance(param="turbidity"), Exceedance(param="coliform"))
    def turbidity_and_coliform(self):
        self.inferences.append(("Turbidity_raises_microbial_risk",
            "High turbidity next to coliform presence raises the microbial risk, because particles shield microbes from disinfection."))
        self.trace.append("R7  turbidity high AND coliform present -> particles shield microbes")

    @Rule(Exceedance(param="ph"), Exceedance(param="lead"))
    def low_ph_and_lead(self):
        self.inferences.append(("Corrosion_driven_lead",
            "pH out of range together with high lead suggests corrosion is leaching lead from the plumbing; fix corrosion control at the root."))
        self.trace.append("R8  pH out of range AND lead high -> corrosion-driven lead leaching")


PRESENCE_KEYS = {"coliform"}


def limit_text(f):
    if f.direction == "band": return "%s - %s" % (f.band_low, f.band_high)
    if f.direction == "min": return ">= %s %s" % (f.limit, f.unit)
    return "<= %s %s" % (f.limit, f.unit)


def diagnose(readings):
    e = AquaRules(); e.reset()
    used = {}
    for param, value in readings.items():
        if param not in KB: continue
        used[param] = value
        if param in PRESENCE_KEYS:
            e.declare(Presence(param=param, present=bool(value)))
        else:
            e.declare(Reading(param=param, value=float(value)))
    e.run()
    findings = []
    for key in e.exceedances:
        f = KB[key]
        findings.append({"parameter": f.label, "reading": used.get(key),
            "limit": limit_text(f), "contaminant": f.contaminant,
            "health_risk": f.health_effect, "likely_source": f.sources,
            "recommend": f.treatments})
    return {"readings": used, "findings": findings,
            "inferences": [{"name": n, "detail": d} for n, d in e.inferences],
            "trace": list(e.trace), "safe": len(e.exceedances) == 0}


def show(readings):
    r = diagnose(readings)
    print("Readings:", readings, "\n")
    print("VERDICT:", "SAFE - all readings within limits" if r["safe"] else "NOT SAFE")
    print()
    for f in r["findings"]:
        print("- %s = %s   (limit %s)" % (f["parameter"], f["reading"], f["limit"]))
        print("    contaminant : %s" % f["contaminant"])
        print("    health risk : %s" % f["health_risk"])
        print("    source      : %s" % ", ".join(f["likely_source"]))
        print("    treatment   : %s" % ", ".join(f["recommend"]))
        print()
    if r["inferences"]:
        print("Combined inferences:")
        for i in r["inferences"]:
            print("  * %s: %s" % (i["name"], i["detail"]))
        print()
    print("Explanation trace (rules that fired):")
    for t in r["trace"]:
        print("  " + t)
    return r


def treatments_for(q):
    q = q.lower()
    for f in KB.values():
        if q in f.key.lower() or q in f.contaminant.lower() or q in f.label.lower():
            return f.treatments
    return []


def sources_for(q):
    q = q.lower()
    for f in KB.values():
        if q in f.key.lower() or q in f.contaminant.lower() or q in f.label.lower():
            return f.sources
    return []


print("AquaReason engine loaded. Knowledge base has", len(KB), "parameters.")


AquaReason engine loaded. Knowledge base has 8 parameters.


In [10]:
#@title  Diagnose your own water sample  { display-mode: "form" }
#@markdown Type the readings, The result appears below.
nitrate_mg_L    = 62    #@param {type:"number"}
arsenic_mg_L    = 0.004 #@param {type:"number"}
lead_mg_L       = 0.002 #@param {type:"number"}
fluoride_mg_L   = 0.6   #@param {type:"number"}
turbidity_NTU   = 1.2   #@param {type:"number"}
pH              = 7.1   #@param {type:"number"}
chlorine_mg_L   = 0.4   #@param {type:"number"}
coliform_present = False #@param {type:"boolean"}

sample = {
    "nitrate": nitrate_mg_L, "arsenic": arsenic_mg_L, "lead": lead_mg_L,
    "fluoride": fluoride_mg_L, "turbidity": turbidity_NTU, "ph": pH,
    "chlorine": chlorine_mg_L, "coliform": coliform_present,
}
show(sample)


Readings: {'nitrate': 62, 'arsenic': 0.004, 'lead': 0.002, 'fluoride': 0.6, 'turbidity': 1.2, 'ph': 7.1, 'chlorine': 0.4, 'coliform': False} 

VERDICT: NOT SAFE

- Nitrate (as NO3) = 62   (limit <= 50.0 mg/L)
    contaminant : Nitrate_Exceedance
    health risk : Methemoglobinemia (blue-baby syndrome) in infants
    source      : Agricultural_runoff, Septic_leakage, Fertiliser_use
    treatment   : Ion_exchange, Reverse_osmosis, Alternative_supply

Explanation trace (rules that fired):
  R1  nitrate reading 62.0 is above the limit 50.0 -> Nitrate_Exceedance


{'readings': {'nitrate': 62,
  'arsenic': 0.004,
  'lead': 0.002,
  'fluoride': 0.6,
  'turbidity': 1.2,
  'ph': 7.1,
  'chlorine': 0.4,
  'coliform': False},
 'findings': [{'parameter': 'Nitrate (as NO3)',
   'reading': 62,
   'limit': '<= 50.0 mg/L',
   'contaminant': 'Nitrate_Exceedance',
   'health_risk': 'Methemoglobinemia (blue-baby syndrome) in infants',
   'likely_source': ['Agricultural_runoff',
    'Septic_leakage',
    'Fertiliser_use'],
   'recommend': ['Ion_exchange', 'Reverse_osmosis', 'Alternative_supply']}],
 'inferences': [],
 'trace': ['R1  nitrate reading 62.0 is above the limit 50.0 -> Nitrate_Exceedance'],
 'safe': False}

In [11]:


print("========== 1) Rural well, fertiliser runoff ==========")
show({"nitrate": 62, "arsenic": 0.004, "ph": 7.1, "chlorine": 0.4, "coliform": False})

print("\n\n========== 2) EPANET network end (low chlorine + coliform) ==========")
show({"turbidity": 6.5, "chlorine": 0.05, "coliform": True})

print("\n\n========== 3) Old building, corrosion driven lead ==========")
show({"lead": 0.028, "ph": 6.1})

print("\n\n========== 4) Clean municipal supply ==========")
show({"nitrate": 10, "arsenic": 0.003, "lead": 0.002, "fluoride": 0.7,
      "turbidity": 0.4, "ph": 7.5, "chlorine": 0.6, "coliform": False})


========== 1) Rural well, fertiliser runoff ==========
Readings: {'nitrate': 62, 'arsenic': 0.004, 'ph': 7.1, 'chlorine': 0.4, 'coliform': False} 

VERDICT: NOT SAFE

- Nitrate (as NO3) = 62   (limit <= 50.0 mg/L)
    contaminant : Nitrate_Exceedance
    health risk : Methemoglobinemia (blue-baby syndrome) in infants
    source      : Agricultural_runoff, Septic_leakage, Fertiliser_use
    treatment   : Ion_exchange, Reverse_osmosis, Alternative_supply

Explanation trace (rules that fired):
  R1  nitrate reading 62.0 is above the limit 50.0 -> Nitrate_Exceedance


========== 2) EPANET network end (low chlorine + coliform) ==========
Readings: {'turbidity': 6.5, 'chlorine': 0.05, 'coliform': True} 

VERDICT: NOT SAFE

- Total coliform / E. coli = True   (limit <= 0 present/absent)
    contaminant : Microbial_Contamination
    health risk : Risk of gastrointestinal disease from faecal pathogens
    source      : Sewage_ingress, Animal_waste, Biofilm
    treatment   : Disinfection_chlorin

{'readings': {'nitrate': 10,
  'arsenic': 0.003,
  'lead': 0.002,
  'fluoride': 0.7,
  'turbidity': 0.4,
  'ph': 7.5,
  'chlorine': 0.6,
  'coliform': False},
 'findings': [],
 'inferences': [],
 'trace': [],
 'safe': True}

In [12]:
#@title  Ask a direct question  { display-mode: "form" }
# @markdown Type a contaminant name (for example: arsenic, nitrate, lead)
contaminant = "arsenic" #@param {type:"string"}

print("Treatments for '%s': %s" % (contaminant, ", ".join(treatments_for(contaminant)) or "none found"))
print("Likely sources of '%s': %s" % (contaminant, ", ".join(sources_for(contaminant)) or "none found"))


Treatments for 'arsenic': Reverse_osmosis, Activated_alumina, Ion_exchange, Coagulation_filtration
Likely sources of 'arsenic': Natural_geological, Industrial_discharge, Mining_activity


In [13]:
# Evaluation

TEST_CASES = [
 {"name":"nitrate over limit","readings":{"nitrate":55.0},"conts":["Nitrate_Exceedance"],"safe":False},
 {"name":"nitrate just under","readings":{"nitrate":49.0},"conts":[],"safe":True},
 {"name":"arsenic over limit","readings":{"arsenic":0.02},"conts":["Arsenic_Exceedance"],"safe":False},
 {"name":"arsenic at limit","readings":{"arsenic":0.01},"conts":[],"safe":True},
 {"name":"lead over limit","readings":{"lead":0.03},"conts":["Lead_Exceedance"],"safe":False},
 {"name":"fluoride over limit","readings":{"fluoride":2.0},"conts":["Fluoride_Exceedance"],"safe":False},
 {"name":"turbidity over limit","readings":{"turbidity":8.0},"conts":["High_Turbidity"],"safe":False},
 {"name":"ph too low","readings":{"ph":5.9},"conts":["pH_Out_Of_Range"],"safe":False},
 {"name":"ph too high","readings":{"ph":9.2},"conts":["pH_Out_Of_Range"],"safe":False},
 {"name":"ph normal","readings":{"ph":7.4},"conts":[],"safe":True},
 {"name":"chlorine too low","readings":{"chlorine":0.1},"conts":["Low_Chlorine_Residual"],"safe":False},
 {"name":"coliform present","readings":{"coliform":True},"conts":["Microbial_Contamination"],"safe":False},
 {"name":"coliform absent","readings":{"coliform":False},"conts":[],"safe":True},
 {"name":"low chlorine + coliform","readings":{"chlorine":0.05,"coliform":True},
  "conts":["Low_Chlorine_Residual","Microbial_Contamination"],"safe":False},
 {"name":"corrosion driven lead","readings":{"ph":6.0,"lead":0.025},
  "conts":["pH_Out_Of_Range","Lead_Exceedance"],"safe":False},
 {"name":"all clean","readings":{"nitrate":10,"arsenic":0.003,"lead":0.002,"fluoride":0.7,
  "turbidity":0.4,"ph":7.5,"chlorine":0.6,"coliform":False},"conts":[],"safe":True},
]

correct = 0
print("Accuracy on labelled set")
print("-" * 60)
for c in TEST_CASES:
    r = diagnose(c["readings"])
    got = sorted(f["contaminant"] for f in r["findings"])
    ok = (r["safe"] == c["safe"]) and (got == sorted(c["conts"]))
    correct += int(ok)
    print("  [%s] %-24s %s" % ("PASS" if ok else "FAIL", c["name"], ", ".join(got) or "-"))
print("-" * 60)
print("  %d / %d correct (%.0f%%)" % (correct, len(TEST_CASES), 100*correct/len(TEST_CASES)))
print()
print("Knowledge-base coverage: %d / %d parameters (%.0f%%)" % (len(KB), len(KB), 100.0))


Accuracy on labelled set
------------------------------------------------------------
  [PASS] nitrate over limit       Nitrate_Exceedance
  [PASS] nitrate just under       -
  [PASS] arsenic over limit       Arsenic_Exceedance
  [PASS] arsenic at limit         -
  [PASS] lead over limit          Lead_Exceedance
  [PASS] fluoride over limit      Fluoride_Exceedance
  [PASS] turbidity over limit     High_Turbidity
  [PASS] ph too low               pH_Out_Of_Range
  [PASS] ph too high              pH_Out_Of_Range
  [PASS] ph normal                -
  [PASS] chlorine too low         Low_Chlorine_Residual
  [PASS] coliform present         Microbial_Contamination
  [PASS] coliform absent          -
  [PASS] low chlorine + coliform  Low_Chlorine_Residual, Microbial_Contamination
  [PASS] corrosion driven lead    Lead_Exceedance, pH_Out_Of_Range
  [PASS] all clean                -
------------------------------------------------------------
  16 / 16 correct (100%)

Knowledge-base coverage: 8